In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path

def strip_ns(tag):
    return tag.split("}")[-1] if "}" in tag else tag

def parse_tcga_clinical_xml(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()
    record = {}

    def recurse(elem):
        children = list(elem)
        if not children:
            text = (elem.text or "").strip()
            tag = strip_ns(elem.tag)
            if text and (tag not in record or not record[tag]):
                record[tag] = text
        else:
            for c in children:
                recurse(c)

    recurse(root)
    return record

def parse_tcga_folder(folder, histology_label):
    records = []
    xml_files = list(Path(folder).rglob("nationwidechildrens.org_clinical.*.xml"))
    print(f"{folder}: found {len(xml_files)} clinical XML files")
    for f in xml_files:
        try:
            rec = parse_tcga_clinical_xml(f)
            rec["HISTOLOGY"] = histology_label
            rec["SOURCE_FILE"] = f.name
            records.append(rec)
        except Exception as e:
            print(f"Failed to parse {f.name}: {e}")
    return pd.DataFrame(records)

luad_df = parse_tcga_folder("TCGA-LUAD", "LUAD")
lusc_df = parse_tcga_folder("TCGA-LUSC", "LUSC")

tcga_nsclc = pd.concat([luad_df, lusc_df], ignore_index=True)
print(tcga_nsclc.shape)
print(tcga_nsclc["HISTOLOGY"].value_counts())
tcga_nsclc.to_csv("tcga_nsclc_cliniccdal.csv", index=False)

TCGA-LUAD: found 0 clinical XML files
TCGA-LUSC: found 0 clinical XML files
(0, 0)


KeyError: 'HISTOLOGY'